In [ ]:
def calc_rmsf(mda_universe, pdb_name):
    """
    This function calculates the RMSF of the structure, saves a PDB with
    the RMSF written in the B-factor column as well as returns the RMSF matrix
    mda_universe = a mda universe object
    pdb_name = prefix for filenames used in this whole package
    this function is so far not multithreaded #TODO
    """
    #compute average structure
    average = mda.analysis.align.AverageStructure(mda_universe, mda_universe, select="protein and name CA", ref_frame=0)
    ref = average.results.universe

    #align to average structure
    aligner = mda.analysis.align.AlignTraj(mda_universe, ref, select="protein and name CA", filename=f"{pdb_name}_aligned_avg_traj.dcd", in_memory=False).run()
    u = mda.Universe(f"{pdb_name}_final.pdb", f"{pdb_name}_aligned_avg_traj.dcd")

    #calc rmsf
    c_alpha = u.select_atoms("protein and name CA")
    rmsf_result = mda.analysis.rms.RMSF(c_alpha).run()

    #save as PDB
    u.add_TopologyAttr('tempfactors') # add empty attribute for all atoms
    protein = u.select_atoms('protein') # select protein atoms
    for residue, r_value in zip(protein.residues, rmsf_result.results.rmsf):
        residue.atoms.tempfactors = r_value

    u.atoms.write(f"{pdb_name}_rmsf_b_factor.pdb")

    #delete temp dcd file
    os.remove(f"{pdb_name}_aligned_avg_traj.dcd")

    return rmsf_result.results.rmsf

def compute_rmsd_block_BAAD(pdb_filename, traj_filename, start_i, stop_i, selection):
    """
    Compute a row block of the RMSD matrix.
    """
    u = mda.Universe(pdb_filename, traj_filename)
    ref = u.copy()

    n_frames = u.trajectory.n_frames
    rmsd_block = np.zeros((stop_i - start_i, n_frames))

    for i, frame_i in enumerate(range(start_i, stop_i)):
        # Compute RMSD for every frame relative to this reference.
        rmsd_analysis = mda.analysis.rms.RMSD(u, ref, selection = selection, superposition = False, ref_frame=frame_i).run()

        # Each row in rmsd_analysis.results.rmsd contains [frame, time, rmsd]
        rmsd_block[i, :] = rmsd_analysis.results.rmsd[:, 2]
    return rmsd_block


def parallel_2d_rmsd_BAAAD(pdb_filename, traj_filename, selection, n_jobs=4):
    """
    Compute the full 2D RMSD matrix in parallel by splitting it into row blocks.
    Parameters:
        pdb_filename (str): Path to the topology file.
        traj_filename (str): Path to the trajectory file.
        n_jobs (int): Number of parallel blocks to use.
    Returns:
        np.ndarray: The computed RMSD matrix.
    """
    u = mda.Universe(pdb_filename, traj_filename)
    n_frames = u.trajectory.n_frames
    frames_per_block = n_frames // n_jobs

    tasks = []
    for i in range(n_jobs):
        start_i = i * frames_per_block
        stop_i = (i + 1) * frames_per_block if i < n_jobs - 1 else n_frames
        tasks.append((pdb_filename, traj_filename, start_i, stop_i, selection))

    # Run RMSD computation in parallel and collect results
    with mp.Pool(n_jobs) as pool:
        results = pool.starmap(compute_rmsd_block, tasks)

    # Assemble the full RMSD matrix in memory
    rmsd_matrix = np.vstack(results)

    print(f"RMSD matrix computation for {selection} complete.")
    return rmsd_matrix


def RMSD_dist_2d(universe, selection):
    """Calculate the RMSD between all frames in a matrix.

    Parameters
    ----------
    universe: MDAnalysis.core.universe.Universe
        MDAnalysis universe.
    selection: str
        Selection string for the atomgroup to be investigated, also used during alignment.

    Returns
    -------
    array: np.ndarray
        Numpy array of RMSD values.
    """
    pairwise_rmsd = mda.analysis.diffusionmap.DistanceMatrix(universe, select=selection).run()
    return pairwise_rmsd.results.dist_matrix

def compute_radgyr_for_frame(ts, protein, masses, total_mass):
    """
    Computes the radius of gyration for a single frame.
    
    Parameters:
        ts: A trajectory frame containing positions.
        protein (AtomGroup): The protein atom group.
        masses (np.ndarray): The masses of the protein atoms.
        total_mass (float): The sum of all masses.
        
    Returns:
        float: The radius of gyration for the frame.
    """
    # Calculate the center of mass for this frame using the provided masses
    com = (protein.positions * masses[:, np.newaxis]).sum(axis=0) / total_mass
    # Compute squared distances of atoms from the center of mass
    diffs = protein.positions - com
    sq_dists = np.sum(diffs**2, axis=1)
    # Return the square root of the mass-weighted average of squared distances
    return np.sqrt(np.sum(masses * sq_dists) / total_mass)

@dask.delayed
def radgyr_process_block(universe, block_slice, protein, masses, total_mass):
    """
    Processes a block of trajectory frames to compute the radius of gyration for each frame.
    
    Parameters:
        universe (MDAnalysis.Universe): The Universe with the trajectory.
        block_slice (slice): A slice object defining the block boundaries.
        protein (AtomGroup): The protein atom group.
        masses (np.ndarray): The masses of the protein atoms.
        total_mass (float): The sum of the masses.
    
    Returns:
        list: A list of radgyr values computed for each frame in the block.
    """
    results = []
    for ts in universe.trajectory[block_slice]:
        radgyr_value = compute_radgyr_for_frame(ts, protein, masses, total_mass)
        results.append(radgyr_value)
    return results

def compute_radgyr_parallel(universe):
    """
    Computes the radius of gyration for each frame in parallel using dask.
    
    Parameters:
        universe (MDAnalysis.Universe): The Universe containing the trajectory.
        n_jobs (int): The number of parallel jobs (blocks) to split the trajectory into.
        
    Returns:
        np.ndarray: An array of computed radgyr values for all frames.
    """
    # Select the protein and prepare mass information
    protein = universe.select_atoms("protein")
    masses = protein.masses
    total_mass = np.sum(masses)
    n_frames = universe.trajectory.n_frames

    # Define blocks using slices so that every frame is processed
    frames_per_block = n_frames // n_jobs
    blocks = []
    for i in range(n_jobs):
        start = i * frames_per_block
        # Last block gets any remaining frames
        stop = (i + 1) * frames_per_block if i < n_jobs - 1 else n_frames
        blocks.append(slice(start, stop))

    # Create a list of delayed tasks for each block
    delayed_tasks = [radgyr_process_block(universe, block, protein, masses, total_mass) for block in blocks]
    # Compute all tasks in parallel
    results = dask.compute(*delayed_tasks)
    # Flatten the list of results into a single numpy array
    radgyr_values = np.concatenate(results)
    return radgyr_values